# agentadapter — 1-Click Colab Launcher

Runs **OpenCode** and **Claude Code** on Google Colab, both wired to the **NVIDIA NIM** API.

**What you need:** an NVIDIA API key from [build.nvidia.com](https://build.nvidia.com) (free tier works).
**Runtime:** CPU is sufficient — no GPU needed.

## Run
1. *(Private repos only)* Add a Colab secret named `GH_TOKEN` containing a GitHub Personal Access Token (🔑 sidebar → *Add secret*).
2. Edit `REPO_URL` in the bootstrap cell below with your GitHub username.
3. Run the bootstrap cell — it clones this repo, installs everything, and runs health checks.
4. Do your work inside `/content/workspace`.

## Keep your work
| Cell | Action |
|------|--------|
| Save to Drive | zips workspace + Claude history → `MyDrive/agentadapter-sessions/` — resumable in any later session |
| Restore from Drive | picks a saved session and unpacks it back |
| Download | zips the workspace straight to your device |

In [ ]:
# ============================================================
# 1-CLICK BOOTSTRAP — clone repo → install deps → provision tools
# Safe to re-run (pulls latest, repairs install).
# ============================================================
import os
import subprocess
import getpass
from pathlib import Path

REPO_URL = "https://github.com/<your-username>/agentadapter.git"  # ← EDIT: your GitHub username
REPO_DIR = "/content/agentadapter"

if "<your-username>" in REPO_URL:
    raise SystemExit("✖ Edit REPO_URL above with your GitHub username, then re-run this cell.")

# --- Optional private-repo access: add a Colab secret named GH_TOKEN ---
token = None
try:
    from google.colab import userdata
    try:
        token = userdata.get("GH_TOKEN")
        print("✔ GH_TOKEN found in Colab Secrets — private clone enabled.")
    except Exception:
        print("ℹ No GH_TOKEN secret — trying a public clone.")
except ImportError:
    print("ℹ Not running in Colab — continuing without Colab Secrets.")

# --- Clone (or update) the repository ---
if Path(REPO_DIR).is_dir():
    print(f"ℹ {REPO_DIR} exists — updating with git pull…")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
else:
    cmd = ["git", "clone"]
    if token:
        helper = f"!f() {{ echo username=x-access-token; echo password={token}; }}; f"
        cmd += ["-c", f"credential.helper={helper}"]
    cmd += [REPO_URL, REPO_DIR]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print((result.stdout or result.stderr).strip())
    if result.returncode != 0:
        raise SystemExit("✖ Clone failed — check REPO_URL (and GH_TOKEN if the repo is private).")

os.chdir(REPO_DIR)
print(f"✔ Repository ready at {REPO_DIR}")

# --- Python dependencies ---
%pip install -q -r requirements.txt

# --- NVIDIA API key (hidden prompt; shared by OpenCode and the NIM proxy) ---
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("🔑 Paste your NVIDIA_API_KEY (input hidden): ")

# --- Provision OpenCode + NIM proxy + Claude Code (idempotent) ---
!python colab_setup.py

# --- Make CLI tools visible to later cells (subprocess PATH tweaks don't persist) ---
for extra in [str(Path.home() / ".local" / "bin"), str(Path.home() / ".opencode" / "bin")]:
    if extra not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{extra}:{os.environ.get('PATH', '')}"
print("✔ Bootstrap complete — CLI tools are on PATH for this session.")

In [ ]:
# 💾 SAVE — zip /content/workspace + Claude history to Google Drive
from colab_workspace import save_snapshot

save_snapshot()  # optional: save_snapshot("descriptive-label")

In [ ]:
# ♻️ RESTORE — pick a saved session from Drive and resume where you left off
from colab_workspace import restore_snapshot

restore_snapshot()

In [ ]:
# ⬇️ DOWNLOAD — grab a zip of the current workspace in your browser
from colab_workspace import download_snapshot

download_snapshot()